# SPK-1 ⭐ — Data / Partition Skew

**Break → Detect → Fix → Prove.** One key holds 90% of the rows, so one task does almost all
the work while the rest sit idle — the most common Spark performance failure.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch it while the cells run.

**Laptop-safe:** the data is *generated lazily* and only `count()`-ed — never collected or
written — so nothing fills memory or disk. Skew is a task-time problem, not a memory problem, so
the default **tuned** box is fine (no need for `make up-constrained`). Nothing to delete at the end.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [1]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import skewed_keys, key_dimension
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run — skew shows up in Stages -> Tasks -> Summary Metrics.
print("Spark UI:", "http://localhost:4040")
spark

Spark UI: http://localhost:4040


## Step 0 — Parameters & the skewed dataset

We synthesize an `orders` fact skewed by `customer_id` (one "house account", `customer_id = 0`,
holds 90% of orders) and a small `customers` dimension — all lazily, nothing stored.

In [2]:
# Lower N_ROWS to 5-10M if your laptop is slow; raise it to make the straggler more dramatic.
N_ROWS       = 20_000_000
N_COLD_KEYS  = 2_000      # "normal" customers the other 10% of orders spread across
HOT_KEY      = 0          # the marketplace "house account"
HOT_FRACTION = 0.90

fact = skewed_keys(spark, n_rows=N_ROWS, hot_key_fraction=HOT_FRACTION,
                   n_cold_keys=N_COLD_KEYS, hot_key=HOT_KEY, key_col="customer_id")
dim  = key_dimension(spark, n_keys=N_COLD_KEYS + 1, key_col="customer_id")

print(f"fact: ~{N_ROWS:,} orders (lazy)    dim: {N_COLD_KEYS + 1:,} customers")

fact: ~20,000,000 orders (lazy)    dim: 2,001 customers


In [3]:
# Confirm the skew — the hot key dominates (small result, safe to show):
(fact.groupBy("customer_id").count()
     .orderBy(F.desc("count"))
     .limit(5)
     .show())

+-----------+--------+
|customer_id|   count|
+-----------+--------+
|          0|17999905|
|        580|    1114|
|        598|    1103|
|        720|    1096|
|       1893|    1083|
+-----------+--------+



## 1. Break it — skewed sort-merge join

The `constrained` session profile turns **AQE off** (no runtime skew rescue) and **broadcast off**
(`autoBroadcastJoinThreshold = -1`), forcing a **sort-merge join** that shuffles both sides by
`customer_id`. Every hot-key row lands in the same reduce partition -> one straggler task.

In [21]:
apply_profile(spark, "constrained")
profile_summary(spark)

broken_join = fact.join(dim, on="customer_id", how="inner")
m_broken = measure(spark, "skewed (AQE off, SMJ)", lambda: broken_join.count())
print("\nbroken-join metrics:", m_broken)

Applied 'constrained' session profile:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 16
Current session knobs:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 16

broken-join metrics: {'label': 'skewed (AQE off, SMJ)', 'runtime_s': 4.2, 'num_tasks': 33, 'shuffle_read_bytes': 10741887, 'shuffle_write_bytes': 10741887, 'spill_mem_bytes': 343932272, 'spill_disk_bytes': 2439913, 'task_time_median_ms': 43.0, 'task_time_max_ms': 3748.0, 'skew_ratio': 87.2, 'peak_mem_max_bytes': 889257312.0}


## 2. Detect it — read the Spark UI

http://localhost:4040 -> **SQL / DataFrame** -> click this query -> the join's reduce **Stage** ->
**Tasks -> Summary Metrics**. The tell: **Max task time >> Median**, and one task with a huge
**Shuffle Read** (often **Spill** too). The cell below prints the same thing as a number —
`skew_ratio = max / median`.

In [22]:
print(f"max task time    : {m_broken['task_time_max_ms']} ms")
print(f"median task time : {m_broken['task_time_median_ms']} ms")
print(f"SKEW RATIO       : {m_broken['skew_ratio']}x   <- one straggler doing ~all the work")

max task time    : 3748.0 ms
median task time : 43.0 ms
SKEW RATIO       : 87.2x   <- one straggler doing ~all the work


## 3. Fix it (A) — Salting

Spread the hot key across `S` partitions: add a random `salt` to the fact, replicate the
dimension across all salts, and join on `(customer_id, salt)`. Still AQE/broadcast off, so this
isolates the salting effect.

In [23]:
S = 16
salted_fact = fact.withColumn("salt", F.floor(F.rand(seed=1) * S).cast("int"))
salted_dim  = dim.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(S)])))
salted_join = salted_fact.join(salted_dim, on=["customer_id", "salt"], how="inner")

m_salt = measure(spark, "salted", lambda: salted_join.count())
print("salted-join metrics:", m_salt)

salted-join metrics: {'label': 'salted', 'runtime_s': 1.81, 'num_tasks': 33, 'shuffle_read_bytes': 50943310, 'shuffle_write_bytes': 50943310, 'spill_mem_bytes': 125828880, 'spill_disk_bytes': 907069, 'task_time_median_ms': 422.0, 'task_time_max_ms': 1371.0, 'skew_ratio': 3.2, 'peak_mem_max_bytes': 142671728.0}


## 3. Fix it (B) — AQE skew-join

Let Spark split the skewed partition at runtime. We keep broadcast **off** (so it stays a
sort-merge join) and **lower the skew threshold** — our data is tiny, so AQE's 256 MB default
would never trip on a laptop. On real (large) data the defaults work as-is.

In [24]:
apply_profile(spark, "tuned", **{
    "spark.sql.autoBroadcastJoinThreshold": "-1",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "16m",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "2",
})
profile_summary(spark)

aqe_join = fact.join(dim, on="customer_id", how="inner")
m_aqe = measure(spark, "AQE skew-join", lambda: aqe_join.count())
print("AQE skew-join metrics:", m_aqe)

Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 200
  spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 16m
  spark.sql.adaptive.skewJoin.skewedPartitionFactor = 2
Current session knobs:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 200
AQE skew-join metrics: {'label': 'AQE skew-join', 'runtime_s': 4.65, 'num_tasks': 24, 'shuffle_read_bytes': 9313814, 'shuffle_write_bytes': 9313814, 'spill_mem_bytes': 352320864, 'spill_disk_bytes': 2439791, 'task_time_median_ms': 54.0, 'task_tim

## 3. Fix it (C) — Broadcast the small side

If one side fits in memory, broadcast it: no shuffle of the huge fact at all, so skew is moot.
Usually the cheapest fix — reach for it first when a side is small.

In [25]:
apply_profile(spark, "tuned")   # broadcast enabled (10 MB threshold)
bcast_join = fact.join(F.broadcast(dim), on="customer_id", how="inner")
m_bcast = measure(spark, "broadcast", lambda: bcast_join.count())
print("broadcast-join metrics:", m_bcast)

Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = 10485760
  spark.sql.shuffle.partitions                     = 200
broadcast-join metrics: {'label': 'broadcast', 'runtime_s': 0.37, 'num_tasks': 17, 'shuffle_read_bytes': 472, 'shuffle_write_bytes': 472, 'spill_mem_bytes': 0, 'spill_disk_bytes': 0, 'task_time_median_ms': 6.0, 'task_time_max_ms': 6.0, 'skew_ratio': 1.0, 'peak_mem_max_bytes': 0.0}


## 4. Prove it

Before/after. Watch the **Skew (max / median)** row collapse from tens-of-x toward ~1x, and
runtime / max-task-time / spill fall with it.

In [26]:
compare([m_broken, m_salt, m_aqe, m_bcast])

| Metric              | skewed (AQE off, SMJ) |       salted | AQE skew-join |    broadcast |
| ------------------- | --------------------- | ------------ | ------------- | ------------ |
| Wall-clock runtime  |                4.20 s |       1.81 s |        4.65 s |       0.37 s |
| Tasks               |                    33 |           33 |            24 |           17 |
| Shuffle read        |               10.2 MB |      48.6 MB |        8.9 MB |      472.0 B |
| Shuffle write       |               10.2 MB |      48.6 MB |        8.9 MB |      472.0 B |
| Spill (memory)      |              328.0 MB |     120.0 MB |      336.0 MB |          0 B |
| Spill (disk)        |                2.3 MB |     885.8 KB |        2.3 MB |          0 B |
| Task time — median  |                 43 ms |       422 ms |         54 ms |         6 ms |
| Task time — max     |              3,748 ms |     1,371 ms |      4,217 ms |         6 ms |
| Skew (max ÷ median) |                 87.2× |         3.2×

## Takeaways & "in real production..."

- **Detect** skew by the task-time **max-vs-median** spread and one fat **Shuffle Read** task —
  averages hide the straggler.
- **Prefer broadcast** when one side fits; else **AQE skew-join** (on by default in Spark 3.2+/4.x);
  fall back to **salting** for explicit control.
- **The "shrink the box" trick:** our data is tiny, so we lowered AQE's skew threshold to
  reproduce the behavior — on real data the defaults trip on their own.
- **In prod:** alert on per-stage task-time skew (max/median), keep AQE enabled, set
  `autoBroadcastJoinThreshold` deliberately, and pre-aggregate / isolate known mega-keys.

## Teardown

Nothing was written (we only counted generated data), so there is nothing to delete. We just
restore the production-tuned safety nets.

In [27]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")

Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = 10485760
  spark.sql.shuffle.partitions                     = 200
Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.
